# IK optimization sweep: effective lengths d1, d2, d3

This notebook sweeps the three **effective solver lengths directly** while keeping the base anchor geometry fixed:

- `d1` = platform offset: `1` to `15` mm, step `0.5` mm
- `d2` = SR link length (historical `d4`): `1` to `15` mm, step `0.5` mm
- `d3` = RR link length: `1` to `15` mm, step `0.5` mm

That is `29 * 29 * 29 = 24,389` geometries.

The **reference design** is marked for comparison at `d1 = 4.0`, `d2 = 11.0` (historical `d4`), `d3 = 9.5` mm — these lie on the 0.5 mm grid, so the reference design is also one of the swept configs.

Acceptance uses the largest connected all-legs-valid workspace:

- width X `>= 8 mm`
- depth Y `>= 8 mm`

No height-span threshold is applied.

This run produces the **sweep table + overview summary only** (the workspace of the reference design vs the swept space). The optional 8-plots-per-config stage is left off.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "kinematics" / "unified_ik_starter.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing kinematics/unified_ik_starter.py")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
IK_DIR = REPO_ROOT / "validetion" / "IK_optimizetion" / "limits"
if str(IK_DIR) not in sys.path:
    sys.path.insert(0, str(IK_DIR))

from ik_optimization_utils import (
    config_label,
    evaluate_workspace,
    inclusive_range,
    save_full_plot_set,
    save_overview_plots,
)

RESULTS_ROOT = IK_DIR / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository root:", REPO_ROOT)
print("Results root:", RESULTS_ROOT)

## Configuration

The solver uses effective lengths `d1`, `d2`, and `d3`. This notebook sweeps **those effective lengths directly** (no physical→effective scaling), leaving the base anchors unchanged.

- `d1` -> platform offset
- `d2` -> SR link length (historical `d4`)
- `d3` -> RR link length

Configured reference design is marked in the summary. The solver base lengths live in `kinematics/unified_ik_starter.py:default_model()`; this notebook overrides them directly with the swept d-values, so only the anchor geometry is taken from the base.

In [ ]:
# Self-contained bootstrap so this configuration cell can run after a kernel restart.
import json
import math
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np

if "REPO_ROOT" not in globals():
    def find_repo_root(start: Path) -> Path:
        start = start.resolve()
        for path in [start, *start.parents]:
            if (path / "kinematics" / "unified_ik_starter.py").exists():
                return path
        raise FileNotFoundError("Could not find repo root containing kinematics/unified_ik_starter.py")
    REPO_ROOT = find_repo_root(Path.cwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

if "IK_DIR" not in globals():
    IK_DIR = REPO_ROOT / "validetion" / "IK_optimizetion" / "limits"
if str(IK_DIR) not in sys.path:
    sys.path.insert(0, str(IK_DIR))

from ik_optimization_utils import (
    FINAL_RELATIVE_PHI456_LIMITS_DEG,
    FINAL_RELATIVE_PHI456_RUN_LABEL,
    atomic_write_json,
    inclusive_range,
    resolve_continue_run_dir,
)

if "RESULTS_ROOT" not in globals():
    RESULTS_ROOT = IK_DIR / "results"
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# =====================================================================
# RUN CONTROL -- edit these to start fresh or resume after a crash.
# =====================================================================
# RUN_MODE = "new"      -> start a brand-new timestamped run from scratch.
# RUN_MODE = "continue" -> resume an existing run, skipping work already done.
#                          The sweep AND the plot stage resume per-configuration,
#                          so you can compute the sweep once and then render the
#                          graphs later without recomputing anything.
RUN_MODE = "new"
# Which run to resume when RUN_MODE == "continue":
#   None            -> the most recent run_* directory (default).
#   "run_2026..."   -> a specific run directory name.
CONTINUE_RUN_ID = None
# Parallel worker processes for the sweep and the plot stage.
MAX_WORKERS = min(8, (os.cpu_count() or 4))

# ---- Plot output (applies in BOTH new and continue modes) ----------
# PLOT_SCOPE chooses which configs get the full 8-figure graph set:
#   "accepted" -> every config that passes acceptance (width>=8, depth>=8 only)
#   "all"      -> every swept config (very large: ~8 PNGs * N_CONFIGS)
#   "subset"   -> only the reference design + top-10 suggestions (fast preview)
# These are rendering choices, so you can change them on a resumed run.
PLOT_SCOPE = "accepted"
MAX_FULL_PLOT_CONFIGS = None   # cap how many configs get plotted (None = no cap)
PLOT_DPI = 150
# Optional visual-only XY refinement. None keeps the sweep grid; 41 gives
# a denser XY picture without changing the scored sweep table.
XY_PROJECTION_POINTS = 41
# =====================================================================

if RUN_MODE not in {"new", "continue"}:
    raise ValueError(f"RUN_MODE must be 'new' or 'continue', got {RUN_MODE!r}")
if PLOT_SCOPE not in {"accepted", "all", "subset"}:
    raise ValueError(f"PLOT_SCOPE must be 'accepted', 'all', or 'subset', got {PLOT_SCOPE!r}")

if RUN_MODE == "continue":
    # ---- Resume: reuse the existing run directory and its frozen sweep config ----
    RUN_DIR = resolve_continue_run_dir(RESULTS_ROOT, CONTINUE_RUN_ID)
    RUN_ID = RUN_DIR.name
    config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))

    D1_MIN_MM, D1_MAX_MM, D1_STEP_MM = config["d1_range_mm"]
    D2_MIN_MM, D2_MAX_MM, D2_STEP_MM = config["d2_range_mm"]
    D3_MIN_MM, D3_MAX_MM, D3_STEP_MM = config["d3_range_mm"]

    _cur = config["current_values_mm"]
    CURRENT_D1_MM = _cur["d1"]
    CURRENT_D2_MM = _cur["d2"]
    CURRENT_D3_MM = _cur["d3"]

    _probe = config["workspace_probe"]
    XY_LIMIT_MM = _probe["xy_limit_mm"]
    Z_MIN_MM = _probe["z_min_mm"]
    Z_MAX_MM = _probe["z_max_mm"]
    XY_POINTS = _probe["xy_points"]
    Z_POINTS = _probe["z_points"]
    PHI1_RAD = _probe["phi1_rad"]

    _acc = config["acceptance"]
    MIN_WORKSPACE_WIDTH_MM = _acc["min_workspace_width_mm"]
    MIN_WORKSPACE_DEPTH_MM = _acc["min_workspace_depth_mm"]
    config.setdefault("movement_restriction", {
        "enabled": True,
        "angles": ["phi4", "phi5", "phi6"],
        "limit_deg": 30.0,
        "angle_frame": "rest_link_local",
        "branch_selection": "angle_valid",
        "relative_limits_deg": FINAL_RELATIVE_PHI456_LIMITS_DEG,
        "restriction_label": FINAL_RELATIVE_PHI456_RUN_LABEL,
        "reference": "per-geometry nominal rest pose P1=(0,0,min(6,d3-0.1)); same phi1_rad",
    })

    print("CONTINUING existing run:", RUN_DIR)
else:
    # ---- New run: sweep the effective solver lengths d1, d2, d3 directly ----
    # d1 = platform offset, d2 = SR link length (historical d4), d3 = RR link.
    # This runs the full link-length sweep, matching validetion/IK_optimizetion,
    # with only the phi4/phi5/phi6 movement restriction changed.
    D1_MIN_MM, D1_MAX_MM, D1_STEP_MM = 1.0, 15.0, 0.5
    D2_MIN_MM, D2_MAX_MM, D2_STEP_MM = 1.0, 15.0, 0.5
    D3_MIN_MM, D3_MAX_MM, D3_STEP_MM = 1.0, 15.0, 0.5

    # Configured reference design values (on the 0.5 mm grid, so also a swept config).
    CURRENT_D1_MM = 4.0    # platform offset d1
    CURRENT_D2_MM = 11.0   # SR link length d2 (historical d4)
    CURRENT_D3_MM = 9.5    # RR link length d3

    # Real deterministic workspace grid.
    # Increase XY_POINTS/Z_POINTS for finer results; this also makes every plot denser and slower.
    XY_LIMIT_MM = 18.0
    Z_MIN_MM = 0.0
    Z_MAX_MM = 18.0
    XY_POINTS = 21
    Z_POINTS = 73
    # Fixed platform/tactor orientation relative to the base frame.
    # No orientation sweep is performed: every workspace point uses this phi1.
    PLATFORM_ORIENTATION_MODE = "fixed"
    PHI1_RAD = math.pi / 2.0

    MIN_WORKSPACE_WIDTH_MM = 8.0
    MIN_WORKSPACE_DEPTH_MM = 8.0

# ---- Derived ranges (identical in both modes) ----
d1_values = inclusive_range(D1_MIN_MM, D1_MAX_MM, D1_STEP_MM)
d2_values = inclusive_range(D2_MIN_MM, D2_MAX_MM, D2_STEP_MM)
d3_values = inclusive_range(D3_MIN_MM, D3_MAX_MM, D3_STEP_MM)
assert np.allclose(np.diff(d1_values), D1_STEP_MM)
assert np.allclose(np.diff(d2_values), D2_STEP_MM)
assert np.allclose(np.diff(d3_values), D3_STEP_MM)
N_CONFIGS = len(d1_values) * len(d2_values) * len(d3_values)

if RUN_MODE == "new":
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_ID = (
        f"run_{timestamp}_{N_CONFIGS}configs_"
        f"d1{D1_MIN_MM:.1f}-{D1_MAX_MM:.1f}_"
        f"d2{D2_MIN_MM:.1f}-{D2_MAX_MM:.1f}_"
        f"d3{D3_MIN_MM:.1f}-{D3_MAX_MM:.1f}_"
        f"{FINAL_RELATIVE_PHI456_RUN_LABEL}"
    ).replace(".", "p")
    RUN_DIR = RESULTS_ROOT / RUN_ID
    config = {
        "d1_range_mm": [D1_MIN_MM, D1_MAX_MM, D1_STEP_MM],
        "d2_range_mm": [D2_MIN_MM, D2_MAX_MM, D2_STEP_MM],
        "d3_range_mm": [D3_MIN_MM, D3_MAX_MM, D3_STEP_MM],
        "number_of_combinations": int(N_CONFIGS),
        "current_values_mm": {"d1": CURRENT_D1_MM, "d2": CURRENT_D2_MM, "d3": CURRENT_D3_MM},
        "workspace_probe": {
            "xy_limit_mm": XY_LIMIT_MM,
            "z_min_mm": Z_MIN_MM,
            "z_max_mm": Z_MAX_MM,
            "xy_points": XY_POINTS,
            "z_points": Z_POINTS,
            "phi1_rad": PHI1_RAD,
            "orientation_mode": PLATFORM_ORIENTATION_MODE if "PLATFORM_ORIENTATION_MODE" in globals() else "fixed",
        },
        "acceptance": {
            "min_workspace_width_mm": MIN_WORKSPACE_WIDTH_MM,
            "min_workspace_depth_mm": MIN_WORKSPACE_DEPTH_MM,
        },
        "movement_restriction": {
            "enabled": True,
            "angles": ["phi4", "phi5", "phi6"],
            "limit_deg": 30.0,
            "angle_frame": "rest_link_local",
            "branch_selection": "angle_valid",
            "relative_limits_deg": FINAL_RELATIVE_PHI456_LIMITS_DEG,
            "restriction_label": FINAL_RELATIVE_PHI456_RUN_LABEL,
            "reference": "per-geometry nominal rest pose P1=(0,0,min(6,d3-0.1)); same phi1_rad",
        },
    }

# Plot settings live in the config dict (used by the renderers) but are taken from
# the top-level RUN CONTROL knobs in BOTH modes, so a resumed run can change them.
config["plot_generation"] = {
    "plot_scope": PLOT_SCOPE,
    "max_full_plot_configs": MAX_FULL_PLOT_CONFIGS,
    "plot_dpi": PLOT_DPI,
    "xy_projection_points": XY_PROJECTION_POINTS,
}

# ---- Output folders (recreated/reused safely in both modes) ----
TABLE_DIR = RUN_DIR / "tables"
PLOT_DIR = RUN_DIR / "plots"
OVERVIEW_DIR = PLOT_DIR / "overview"
ALL_CONFIG_DIR = PLOT_DIR / "all_configurations"
for path in [TABLE_DIR, OVERVIEW_DIR, ALL_CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Persist the frozen config only for a new run; a resumed run keeps its original
# sweep grid/probe untouched (we only changed in-memory plot settings above).
if RUN_MODE == "new":
    atomic_write_json(RUN_DIR / "run_config.json", config)

print("Run mode:", RUN_MODE)
print("Run directory:", RUN_DIR)
print("Parallel workers:", MAX_WORKERS)
print("Plot scope:", PLOT_SCOPE, "| max plotted:", MAX_FULL_PLOT_CONFIGS, "| dpi:", PLOT_DPI, "| XY plot points:", XY_PROJECTION_POINTS)
print("Movement restriction:", config.get("movement_restriction"))
print("Platform orientation:", config.get("workspace_probe", {}).get("orientation_mode", "fixed"), "phi1_rad=", PHI1_RAD)
print(f"All combinations to test: {N_CONFIGS:,}")
print(f"d1 values ({len(d1_values)}):", d1_values)
print(f"d2 values ({len(d2_values)}):", d2_values)
print(f"d3 values ({len(d3_values)}):", d3_values)
print(f"Reference design marked at: d1={CURRENT_D1_MM}, d2={CURRENT_D2_MM} (hist. d4), d3={CURRENT_D3_MM}")
print("\nTip: while the plot stage runs you can pause Dropbox sync to avoid")
print("file-sync contention on the thousands of image files written here.")

## Run the complete parameter sweep

This evaluates all combinations of `d1`, `d2`, `d3` over the 1?15 mm grid (step 0.5 mm). That is `29 * 29 * 29 = 24,389` geometries. The only difference from `validetion/IK_optimizetion` is the finalized rest-link-local relative `phi4`/`phi5`/`phi6` restriction: top `15/90/40`, right/left `20/90/35`.


In [ ]:
# Parameter sweep -- parallel across CPU cores and crash-resumable.
#
# Each configuration is computed in a worker process and its result is written
# atomically to its own JSON file under tables/partial/ the instant it finishes.
# Re-running this cell (or running it with RUN_MODE="continue") skips any config
# whose partial file already exists, so a crash only costs the configs that were
# actually in flight. The final CSV is assembled from the partial files in the
# original (d1, d2, d3) nesting order, so sweep_results.csv is deterministic
# regardless of completion order.
import json
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from IPython.display import display

from ik_optimization_utils import (
    evaluate_and_checkpoint,
    partial_result_label,
)

PARTIAL_DIR = TABLE_DIR / "partial"
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

all_configs = [(a, b, c) for a in d1_values for b in d2_values for c in d3_values]


def _partial_path(d1, d2, d3):
    return PARTIAL_DIR / f"{partial_result_label(d1, d2, d3)}.json"


todo = [c for c in all_configs if not _partial_path(*c).exists()]
print(
    f"Sweep: {len(all_configs) - len(todo):,} already computed, "
    f"{len(todo):,} to compute  (workers={MAX_WORKERS})"
)

t0 = time.perf_counter()
if todo:
    tasks = [(d1, d2, d3, config, str(PARTIAL_DIR)) for (d1, d2, d3) in todo]
    completed = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(evaluate_and_checkpoint, t) for t in tasks]
        for _ in as_completed(futures):
            completed += 1
            if completed % 50 == 0 or completed == len(tasks):
                elapsed = time.perf_counter() - t0
                rate = completed / elapsed if elapsed > 0 else float("nan")
                remaining = (len(tasks) - completed) / rate if rate > 0 else float("nan")
                print(f"  {completed:>5,}/{len(tasks):,} done  elapsed={elapsed:,.1f}s  est_remaining={remaining:,.1f}s")
else:
    print("  nothing to do -- all configurations already computed.")

# Assemble the full results table from the per-config checkpoints, reading in
# canonical config order so ordering/tie-breaking is deterministic.
missing = [c for c in all_configs if not _partial_path(*c).exists()]
if missing:
    raise RuntimeError(f"{len(missing)} configurations are still missing checkpoints; re-run this cell to finish them.")
rows = [json.loads(_partial_path(*c).read_text(encoding="utf-8")) for c in all_configs]

results = pd.DataFrame(rows).sort_values(["accepted", "score"], ascending=[False, False]).reset_index(drop=True)
# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
results_path = TABLE_DIR / "sweep_results.csv"
results.to_csv(results_path, index=False)
print("Saved:", results_path)
display(results.head(10))

In [ ]:
from ik_optimization_utils import evaluate_workspace

# Evaluate the configured reference design directly. These values land on
# the 0.5 mm grid, so it is also among the swept configs; evaluating it directly
# guarantees the summary's reference-design marker uses the same exact values.
current_row = pd.Series(
    evaluate_workspace(CURRENT_D1_MM, CURRENT_D2_MM, CURRENT_D3_MM, config, keep_detail=False)
)

accepted = results[results["accepted"]].copy()
suggestions = accepted.copy() if not accepted.empty else results.head(50).copy()

# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
suggestions_path = TABLE_DIR / "suggested_values_all.csv"
suggestions.to_csv(suggestions_path, index=False)
current_path = TABLE_DIR / "current_values_comparison.csv"
pd.DataFrame([current_row.to_dict()]).to_csv(current_path, index=False)

print(f"Accepted suggestion count: {len(accepted):,}")
print(f"Saved all suggested values: {suggestions_path}")
print(f"Reference design row (d1={CURRENT_D1_MM}, d2={CURRENT_D2_MM} hist. d4, d3={CURRENT_D3_MM}):")
display(pd.DataFrame([current_row]))
print("Top suggestions:")
display(suggestions[[
    "d1_mm", "d2_mm", "d3_mm",
    "workspace_width_mm", "workspace_depth_mm", "workspace_area_mm2",
    "max_dimension_mm", "total_envelope_mm", "d2_minus_d3_mm",
    "d2_gt_d3_preference_factor", "workspace_efficiency",
    "valid_fraction", "largest_component_fraction", "score", "accepted"
]].head(20))

In [ ]:
OVERVIEW_DIR.mkdir(parents=True, exist_ok=True)
save_overview_plots(results, current_row, suggestions, config, OVERVIEW_DIR)
print("Overview plots saved under:", OVERVIEW_DIR)

## Save the full graph set per configuration

`PLOT_SCOPE` (set in the config cell) controls which configs get the full eight-figure set:

- `"accepted"` -- every config passing acceptance (width>=8, depth>=8; no height threshold). **Current setting.**
- `"all"` -- every swept config (~8 x N_CONFIGS PNGs; very large).
- `"subset"` — only the reference design + top-10 suggestions (fast preview).


For a clearer XY presentation, `XY_PROJECTION_POINTS` can be set higher than the scored sweep grid. For example, `41` gives 0.9 mm XY spacing instead of 1.8 mm, producing more unique XY points in `workspace_projection_xy.png` without changing the sweep score/table.

Each configuration gets a folder `plots/all_configurations/d1_<value>/d2_<value>/d3_<value>/` containing:

- `workspace_projection_xy.png`, `workspace_projection_xz.png`, `workspace_projection_yz.png`
- `workspace_3d_per_leg_and_all.png`
- `angle_distributions.png`, `angle_deviation_from_mean.png`
- `relative_error_boxplot.png`, `relative_error_mean_max.png`
- `workspace_summary.json`

This stage is **crash-resumable**: a `.done` marker is written only after the configured PNGs flush, so re-running (or `RUN_MODE="continue"`) skips finished configs and renders only what's left. With `PLOT_SCOPE="accepted"`, this can be many images -- pause Dropbox sync while it runs.


In [ ]:
# Full graph set -- parallel and crash-resumable.
#
# PLOT_SCOPE (set in the config cell) chooses which configs get the eight-figure
# set: "accepted" (configs passing acceptance), "all" (every config), or "subset"
# (reference design + top-10 suggestions). Each config is rendered in a worker
# process; a ".done" marker is written only after all configured PNGs and the summary
# JSON are flushed, so re-running skips any already-complete config -- a crash or
# stop costs only the configs that were mid-render. Safe to resume any time.
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd

from ik_optimization_utils import (
    _config_output_dir,
    plot_config_done,
    render_and_checkpoint,
)

# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
ALL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if PLOT_SCOPE == "all":
    plot_rows = results.copy()
elif PLOT_SCOPE == "accepted":
    # Keep the usual accepted-config plot set, but always include the current
    # design too, even if it fails acceptance under the +/-30 deg restriction.
    plot_rows = pd.concat([results[results["accepted"]].copy(), pd.DataFrame([current_row])], ignore_index=True).drop_duplicates(
        subset=["d1_mm", "d2_mm", "d3_mm"]
    )
else:  # "subset"
    plot_rows = pd.concat([pd.DataFrame([current_row]), suggestions.head(10)], ignore_index=True).drop_duplicates(
        subset=["d1_mm", "d2_mm", "d3_mm"]
    )

if MAX_FULL_PLOT_CONFIGS is not None:
    plot_rows = plot_rows.head(int(MAX_FULL_PLOT_CONFIGS)).copy()

print(f"Plot scope '{PLOT_SCOPE}': {len(plot_rows):,} / {len(results):,} configs get the full 8-figure set")

row_dicts = [row.to_dict() for _, row in plot_rows.iterrows()]
todo = [d for d in row_dicts if not plot_config_done(d, str(ALL_CONFIG_DIR))]
print(
    f"Plots: {len(row_dicts) - len(todo):,} already complete, "
    f"{len(todo):,} to render  (workers={MAX_WORKERS})"
)

t0 = time.perf_counter()
if todo:
    tasks = [(d, config, str(ALL_CONFIG_DIR)) for d in todo]
    completed = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(render_and_checkpoint, t) for t in tasks]
        for _ in as_completed(futures):
            completed += 1
            if completed % 10 == 0 or completed == len(tasks):
                elapsed = time.perf_counter() - t0
                rate = completed / elapsed if elapsed > 0 else float("nan")
                remaining = (len(tasks) - completed) / rate if rate > 0 else float("nan")
                print(f"  plots {completed:>5,}/{len(tasks):,} saved  elapsed={elapsed:,.1f}s  est_remaining={remaining:,.1f}s")
else:
    print("  nothing to do -- all targeted plot sets already complete.")

# Manifest covers every targeted configuration (rendered now or in a prior run).
saved_dirs = [_config_output_dir(d, ALL_CONFIG_DIR) for d in row_dicts]
n_plot_done = sum(plot_config_done(d, str(ALL_CONFIG_DIR)) for d in row_dicts)
plot_manifest = pd.DataFrame({"plot_dir": [str(p) for p in saved_dirs]})
plot_manifest_path = TABLE_DIR / "plot_manifest.csv"
plot_manifest.to_csv(plot_manifest_path, index=False)
print(f"Completed plot-set folders: {n_plot_done:,} / {len(row_dicts):,}")
print("Saved plot manifest:", plot_manifest_path)

## Final recommendation list

The final box prints the configured **reference design** and the ranked suggestions, with the **length of the suggested list** and the effective lengths for every printed suggestion.

**Important modeling note:** the three variables are the effective solver lengths directly (`d1` platform offset, `d2` SR link / historical `d4`, `d3` RR link). Confirm these against CAD or measurements before final fabrication decisions.

In [ ]:
print("RESULTS FOLDER")
print(RUN_DIR)
print()
print("COMBINATION COUNT")
print(f"All tested combinations: {len(results):,}")
print(f"Accepted/suggested combinations: {len(accepted):,}")
print(f"Suggestions table length: {len(suggestions):,}")
print(f"Full plot-set folders complete: {n_plot_done if 'n_plot_done' in globals() else 0:,}")

print()
print("REFERENCE DESIGN VALUES")
print(
    f"d1/platform offset={current_row['d1_mm']:.2f} mm, "
    f"d2/SR link={current_row['d2_mm']:.2f} mm (historical d4), "
    f"d3/RR link={current_row['d3_mm']:.2f} mm | "
    f"workspace XY={current_row['workspace_width_mm']:.2f} x {current_row['workspace_depth_mm']:.2f} mm, "
    f"max dimension max(d1,d2,d3)={current_row['max_dimension_mm']:.2f} mm, "
    f"total envelope d1+d2+d3={current_row['total_envelope_mm']:.2f} mm, "
    f"d2-d3={current_row['d2_minus_d3_mm']:.2f} mm, "
    f"d2>d3 factor={current_row['d2_gt_d3_preference_factor']:.3f}, "
    f"efficiency={current_row['workspace_efficiency']:.4f}, "
    f"accepted={bool(current_row['accepted'])}, score={current_row['score']:.4f}"
)

if accepted.empty:
    print()
    print("No candidate met workspace width X >= 8 mm and depth Y >= 8 mm.")
    print("Best non-accepted candidates from this sweep:")
else:
    print()
    print("SUGGESTED VALUES (accepted candidates, ranked by score)")
    print(f"Number/length of suggested list: {len(suggestions):,}")

for rank, (_, row) in enumerate(suggestions.head(30).iterrows(), start=1):
    print(
        f"{rank:02d}. d1/platform={row['d1_mm']:.2f} mm, "
        f"d2/SR={row['d2_mm']:.2f} mm, d3/RR={row['d3_mm']:.2f} mm | "
        f"workspace={row['workspace_width_mm']:.2f} x {row['workspace_depth_mm']:.2f} mm, "
        f"area={row['workspace_area_mm2']:.2f} mm^2, "
        f"max dimension={row['max_dimension_mm']:.2f} mm, "
        f"total envelope d1+d2+d3={row['total_envelope_mm']:.2f} mm, "
        f"d2-d3={row['d2_minus_d3_mm']:.2f} mm, "
        f"d2>d3 factor={row['d2_gt_d3_preference_factor']:.3f}, "
        f"efficiency={row['workspace_efficiency']:.4f}, "
        f"valid_fraction={row['valid_fraction']:.3f}, "
        f"largest_component_fraction={row['largest_component_fraction']:.3f}, "
        f"score={row['score']:.4f}"
    )